In [1]:
import zipfile
import os

zip_path = "/content/face_dataset.zip"

print(os.path.exists(zip_path))
print(os.path.getsize(zip_path))



with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/face_dataset/")

False


FileNotFoundError: [Errno 2] No such file or directory: '/content/face_dataset.zip'

In [ ]:
!file /content/face_dataset.zip

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [2]:
import zipfile

try:

  with zipfile.ZipFile(zip_path,"r") as zip_ref:
    bad_file=zip_ref.testzip()
    if bad_file is None:
      print("it is a valid zip file")
    else:
      print("it is a corrupt zip file", bad_file)
except Exception as e:
  print(e)


[Errno 2] No such file or directory: '/content/face_dataset.zip'


In [3]:
!unzip -t /content/face_dataset.zip

unzip:  cannot find or open /content/face_dataset.zip, /content/face_dataset.zip.zip or /content/face_dataset.zip.ZIP.


In [4]:
import os

os.listdir("/content/face_dataset/face_dataset/face_dataset")

FileNotFoundError: [Errno 2] No such file or directory: '/content/face_dataset/face_dataset/face_dataset'

In [ ]:


dataset_path = "/content/face_dataset/face_dataset/face_dataset"

folders = os.listdir(dataset_path)

print("Total Persons:", len(folders))
print(folders[:10])

In [ ]:
# import matplotlib.pyplot as plt
images=[]
for folder in folders:


  images_path = f"/content/face_dataset/face_dataset/face_dataset/{folder}"
  img = os.listdir(images_path)
  images.append(img)



X=images
Y=folders

In [ ]:
X

In [ ]:
from torchvision import datasets, transforms
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])





dataset = datasets.ImageFolder(
    root="/content/face_dataset/face_dataset/face_dataset",
    transform=transform
)
dataset

In [5]:
print(len(dataset.classes))
print(dataset.targets)

NameError: name 'dataset' is not defined

In [6]:
from collections import Counter

class_counts = Counter(dataset.targets)

print("People with only 1 image:",
      sum(c == 1 for c in class_counts.values()))

print("People with 2+ images:",
      sum(c >= 2 for c in class_counts.values()))

NameError: name 'dataset' is not defined

In [7]:
dataset.targets

NameError: name 'dataset' is not defined

In [8]:
from collections import Counter


class_counts = Counter(dataset.targets)

valid_classes = set()
for cls, count in class_counts.items():
        if count >= 2:
          valid_classes.add(cls)

valid_indices = [
    i for i, label in enumerate(dataset.targets)
    if label in valid_classes
]

print(len(valid_indices))


NameError: name 'dataset' is not defined

In [9]:
dataset

NameError: name 'dataset' is not defined

In [10]:
from collections import defaultdict

label_to_indices = defaultdict(list)

for idx in valid_indices:

    _, label = dataset[idx]

    label_to_indices[label].append(idx)

print("Total usable people:",
      len(label_to_indices))
label_to_indices

NameError: name 'valid_indices' is not defined

In [ ]:
import random
from torch.utils.data import Dataset


class SiameseDataset(Dataset):

    def __init__(self,
                 dataset,
                 label_to_indices):

        self.dataset = dataset
        self.label_to_indices = label_to_indices

        self.labels = list(
            label_to_indices.keys()
        )

        self.valid_indices = []

        for label in self.labels:
            self.valid_indices.extend(
                label_to_indices[label]
            )

    def __len__(self):

        return len(self.valid_indices)

    def __getitem__(self, index):

    # Select first image
      idx1 = self.valid_indices[index]

      img1, label1 = self.dataset[idx1]

      # Randomly decide whether pair is same or different
      label = random.randint(0, 1)

      if label == 1:

          # Same person pair
          idx2 = idx1

          while idx2 == idx1:
              idx2 = random.choice(
                  self.label_to_indices[label1]
              )

      else:

          # Different person pair
          label2 = label1

          while label2 == label1:
              label2 = random.choice(
                  self.labels
              )

          idx2 = random.choice(
              self.label_to_indices[label2]
          )

      img2, _ = self.dataset[idx2]

      return img1, img2, label






In [11]:
from torch.utils.data import DataLoader
from torch.utils.data import random_split

# Create Siamese Dataset
siamese_dataset = SiameseDataset(
    dataset,
    label_to_indices
)

# Split into Train and Test
train_size = int(0.8 * len(siamese_dataset))
test_size = len(siamese_dataset) - train_size

train_dataset, test_dataset = random_split(
    siamese_dataset,
    [train_size, test_size]
)

# Train Loader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

# Test Loader
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

NameError: name 'SiameseDataset' is not defined

In [ ]:
test_dataset

In [ ]:
img1, img2, label = siamese_dataset[0]

print(img1.shape)
print(img2.shape)
print(label)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(5,2,
                         figsize=(6,15))

for i in range(5):

    img1, img2, label = siamese_dataset[i]

    axes[i,0].imshow(
        img1.permute(1,2,0)
    )

    axes[i,0].axis("off")

    axes[i,1].imshow(
        img2.permute(1,2,0)
    )

    axes[i,1].set_title(
        f"Same: {label}"
    )

    axes[i,1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
from torch import nn

class MYNN(nn.Module):

    def __init__(self, input_features):
        super().__init__()

        # Shared Feature Extractor
        self.features = nn.Sequential(

            nn.Conv2d(input_features, 32, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.2),
        )

        # Similarity Classifier
        self.classifier = nn.Sequential(

            nn.Linear(128 * 16 * 16, 512),
            nn.ReLU(),


            nn.Linear(512, 256),
            nn.ReLU(),


            nn.Linear(256, 128),
            nn.ReLU(),


            nn.Linear(128, 1)
        )

    def forward(self, img1, img2):

        # Extract features from image 1
        feat1 = self.features(img1)

        # Extract features from image 2
        feat2 = self.features(img2)

        # Flatten
        feat1 = torch.flatten(feat1, start_dim=1)
        feat2 = torch.flatten(feat2, start_dim=1)

        # Absolute difference
        diff = torch.abs(feat1 - feat2)

        # Classification
        output = self.classifier(diff)

        return output

In [ ]:
learning_rate = 0.001
epochs = 20

In [ ]:
model = MYNN(input_features=3).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
 # Training Loop
for epoch in range(epochs):

    model.train()
    running_loss = 0.0

    for img1, img2, labels in train_loader:
        img1 = img1.to(device)
        img2 = img2.to(device)

        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(
            img1,
            img2


        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * img1.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    print(
        f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}"
    )

In [ ]:
zip_path = "/content/imageFolder2.zip"




with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/imageFolder2/")

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])





test_dataset = datasets.ImageFolder(
    root="/content/imageFolder2/",
    transform=transform
)
test_dataset

In [ ]:
img1, label1 = test_dataset[0]
img2, label2 = test_dataset[0]
print(label1, label2)

img1 = img1.unsqueeze(0)  # Add batch dimension
img2 = img2.unsqueeze(0)

img1 = img1.to(device)
img2 = img2.to(device)

model.eval()

with torch.no_grad():
    outputs = model(img1, img2)

prob = torch.sigmoid(outputs)
print('same'if prob>=0.5 else 'different')
print("probability", prob)

In [ ]:
print(len(train_loader.dataset))

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for img1, img2, labels in train_loader:

        img1 = img1.to(device)
        img2 = img2.to(device)
        labels = labels.float().to(device)

        outputs = model(img1, img2)

        # Convert logits to probabilities
        probs = torch.sigmoid(outputs)

        # Convert probabilities to 0 or 1
        preds = (probs >= 0.5).float()

        correct += (preds.squeeze() == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total

print(f"Training Accuracy: {accuracy:.2f}%")

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for img1, img2, labels in test_loader:

        img1 = img1.to(device)
        img2 = img2.to(device)

        labels = labels.float().to(device)

        outputs = model(img1, img2)

        probs = torch.sigmoid(outputs)

        preds = (probs >= 0.5).float()

        correct += (preds.squeeze() == labels).sum().item()

        total += labels.size(0)

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
!pip install optuna


In [ ]:
EPOCHS = 20

In [ ]:
import torch
from torch import nn

class MYNN(nn.Module):

    def __init__(
        self,
        input_features,
        conv_dropout,
        fc_dropout,
        hidden_dim,
        use_batchnorm
    ):
        super().__init__()

        bn1 = nn.BatchNorm2d(32) if use_batchnorm else nn.Identity()
        bn2 = nn.BatchNorm2d(64) if use_batchnorm else nn.Identity()
        bn3 = nn.BatchNorm2d(128) if use_batchnorm else nn.Identity()

        self.features = nn.Sequential(

            nn.Conv2d(input_features, 32, 3, padding=1),
            nn.ReLU(),
            bn1,
            nn.MaxPool2d(2),
            nn.Dropout(conv_dropout),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            bn2,
            nn.MaxPool2d(2),
            nn.Dropout(conv_dropout),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            bn3,
            nn.MaxPool2d(2),
            nn.Dropout(conv_dropout),
        )

        self.classifier = nn.Sequential(

            nn.Linear(128 * 16 * 16, hidden_dim),
            nn.ReLU(),
            nn.Dropout(fc_dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(fc_dropout),

            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, img1, img2):

        feat1 = self.features(img1)
        feat2 = self.features(img2)

        feat1 = torch.flatten(feat1, start_dim=1)
        feat2 = torch.flatten(feat2, start_dim=1)

        diff = torch.abs(feat1 - feat2)

        return self.classifier(diff)

In [ ]:
import optuna
from torch.utils.data import DataLoader

def objective(trial):

    lr = trial.suggest_float(
        "lr",
        1e-5,
        1e-2,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [16, 32, 64]
    )

    conv_dropout = trial.suggest_float(
        "conv_dropout",
        0.0,
        0.4
    )

    fc_dropout = trial.suggest_float(
        "fc_dropout",
        0.0,
        0.6
    )

    hidden_dim = trial.suggest_categorical(
        "hidden_dim",
        [128, 256, 512]
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        1e-2,
        log=True
    )

    use_batchnorm = trial.suggest_categorical(
        "use_batchnorm",
        [True, False]
    )

    optimizer_name = trial.suggest_categorical(
        "optimizer",
        ["Adam", "AdamW"]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model = MYNN(
        input_features=3,
        conv_dropout=conv_dropout,
        fc_dropout=fc_dropout,
        hidden_dim=hidden_dim,
        use_batchnorm=use_batchnorm
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
    else:
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

    EPOCHS = 20

    for epoch in range(EPOCHS):

        model.train()

        for img1, img2, labels in train_loader:

            img1 = img1.to(device)
            img2 = img2.to(device)

            labels = (
                labels
                .float()
                .unsqueeze(1)
                .to(device)
            )

            optimizer.zero_grad()

            outputs = model(
                img1,
                img2
            )

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for img1, img2, labels in test_loader:

            img1 = img1.to(device)
            img2 = img2.to(device)

            outputs = model(
                img1,
                img2
            )

            probs = torch.sigmoid(outputs)

            preds = (probs >= 0.5).float()

            correct += (
                preds.squeeze().cpu() == labels
            ).sum().item()

            total += labels.size(0)

    accuracy = correct / total

    return accuracy

In [ ]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=25
)

In [ ]:
test_loader
[I 2026-06-12 15:44:03,918] Trial 2 finished with value: 0.6917621385706492 and parameters: {'lr': 3.422383659907737e-05, 'batch_size': 32, 'conv_dropout': 0.10873759101501386, 'fc_dropout': 0.28291117399072946, 'hidden_dim': 256, 'weight_decay': 8.327734389714001e-05, 'use_batchnorm': True, 'optimizer': 'AdamW'}

